In [1]:
# math librairies
import numpy as np      # see: https://mathesaurus.sourceforge.net/matlab-numpy.html

from plotly import express as px
from plotly import subplots as splt
#from plotly import figure_factory as ff
import pandas               # used with plotly

# dnn
import torch
from torch.nn import MSELoss, Conv2d, ModuleList, Hardtanh
from torch.optim import Adam

# custom librairies 
import imicpe
import imicpe.optim as optim
print(imicpe.__version__)

1.0.16


## Choosing the computation device (CPU/GPU)

The provided `chooseDevice()` function automatically selects the GPU if one is available.  
It is also possible to explicitly specify your preferred device if needed:
- CPU: `device='cpu'`
- NVIDIA or AMD GPU: `device='cuda'`
- Apple M# processor: `device='mps'`

*Note.* A good practice is to **explicitly specify the computation device** when using PyTorch functions whenever possible (by using the `device=device` option), for example when creating the dataset, etc.

In [2]:
device = optim.chooseDevice()

Using device: cpu



# **Initialization**

## Loading datasets (ground truth images)

We use the BSDS500 dataset ([Berkeley Segmentation Data Set and Benchmarks 500](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html)), which contains 500 images, divided into:
- 200 images for training,
- 100 images for validation,
- 200 images for testing.

The provided `BSDSDataset` class allows loading the images and storing them in an object that behaves like a list:
- for a dataset `DS`, its length can be obtained with `len(DS)`,
- an element of the dataset can be accessed using the bracket operator, for example `DS[9]`.

In [3]:
dataset_options = dict(
    root_dir = "BSDS500/BSDS500/images",        # local path towards data base
    image_size = (128, 128),            # desired image size, which will be cropped if necessary
    image_cnt = None,                   # number of images extracted from the data base if specified
    gray = True,
    device = device,
)

train_bsds = optim.BSDSDataset(mode="train", **dataset_options)
val_bsds   = optim.BSDSDataset(mode="val",   **dataset_options)
test_bsds  = optim.BSDSDataset(mode="test",  **dataset_options)

print("Number of images in the training data base :",  len(train_bsds))
print("Number of images in the validation data base :",   len(val_bsds))
print("Number of images in the testing data base :",         len(test_bsds))


Number of images in the training data base : 200
Number of images in the validation data base : 100
Number of images in the testing data base : 200


In [4]:
# example
Nex = 9
xbar_ex, image_id = train_bsds[Nex]

fig = px.imshow(
            optim.torchImg2Numpy(xbar_ex), 
            range_color=[0,1], 
            color_continuous_scale='gray', 
            title="Example of an xbar image in the training data base"
        )
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

## Degraded image base generation

### Implementation of the degradation model (forward model)

From these ground truth images, the degraded data are generated by adding Gaussian noise to each channel of the image (see [`torch.randn`](https://pytorch.org/docs/stable/generated/torch.randn.html)).

In [5]:
def addGaussianNoise(clean_image, sigma, seed=2026):        
    noise = torch.randn(clean_image.shape, device = clean_image.device, dtype = clean_image.dtype, generator=torch.Generator().manual_seed(seed))*sigma
    noisy_image = clean_image + noise

    return noisy_image

### Generation of noisy datasets

The `NoisyDataset` class allows generating a degraded dataset from a ground truth image dataset, and accepts the following arguments:
- `dataset`: the ground truth image dataset,
- `degradation_model`: the function modeling the degradation,
- `sigma`: the standard deviation of the Gaussian noise,
- `seed` (optional): the random number generator seed.

In addition, the `.toTensorDataset()` method allows storing the dataset in memory if desired.

In [6]:
sigma = 0.1
seed = 2025         # for reproducibility

train_noisybsds = optim.NoisyDataset(train_bsds,degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()
val_noisybsds   = optim.NoisyDataset(val_bsds,degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()
test_noisybsds  = optim.NoisyDataset(test_bsds,degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()

print("Number of degraded images in the training data base :",  len(train_noisybsds))
print("Number of degraded images in the validation data base :",   len(val_noisybsds))
print("Number of degraded images in the test data base :",         len(test_noisybsds))


Number of degraded images in the training data base : 200
Number of degraded images in the validation data base : 100
Number of degraded images in the test data base : 200


In [7]:
# exemple (suite)
z_ex, xbar_ex, image_id = train_noisybsds[Nex]

img_xbar_ex = optim.torchImg2Numpy(xbar_ex)
img_z_ex    = optim.torchImg2Numpy(z_ex)

snr_ex = optim.snr(img_z_ex,img_xbar_ex)

fig = px.imshow(np.array([img_xbar_ex,img_z_ex]), color_continuous_scale='gray',
                    title="Example of a pair (xbar,z) from the training data base",
                    facet_col=0, facet_col_spacing=0, facet_col_wrap=2,
                    width=800, height=400,
                    )
fig.update_coloraxes(cmin=0, cmax=1)

legend = ['Ground truth xbar','Degraded data z (PSNR='  +"{:.2f}".format(snr_ex)+'dB)']
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))

fig.show()

# Unrolled ISTA for image denoising

In [8]:
# seed choice to initiate the generator
torch.manual_seed(5117915234194007355)

## Network creation

### Setting up the network architecture

We use convolutional layers, so the PyTorch syntax is as follows:

`torch.nn.Conv2d(input_channel, output_channel, kernel_size, bias=False, padding=1, padding_mode='zeros')`

In [9]:
class UnfoldedISTA(torch.nn.Module):
    def __init__(self, channels=3, features=5, depth=3):
        """ UnfoldedISTA construction

        Parameters
        ----------
        channels: int
            Number of channels in the image
        features: int
            Number of nodes for each layer
        depth: int
            Number of layers in the network
        """
        
        super().__init__()

        # creation of operators
        self.D  = Conv2d(in_channels=channels, out_channels=features, kernel_size=3, padding=1)     # primal to dual operator
        self.Dt = Conv2d(in_channels=features, out_channels=channels, kernel_size=3, padding=1)     # dual to primal operator

        #  creation of weight matrixes and bias vectors
        self.bias_layers   = ModuleList()
        self.weight_layers = ModuleList()
        for i in range(depth):
            self.bias_layers  .append(Conv2d(in_channels=channels, out_channels=features, kernel_size=3, padding=1))
            self.weight_layers.append(Conv2d(in_channels=features, out_channels=features, kernel_size=3, padding=1))

        # activation function
        self.activation = torch.nn.ReLU()

    ### architecture construction 
    def forward(self, z):
        # entry layer (initialization)
        u = self.D(z)

        # hidden layers 
        for bias, weight in zip(self.bias_layers, self.weight_layers):
            u = self.activation(weight(u)-bias(z))
        # exit layer
        x = z-self.Dt(u)
    
        return x


### Network instantiation

In [10]:
# network parameters
Nchannels = 1              # number of channels in the images
Nnodes    = 10              # number of nodes for each layer
Nlayers   = 10               # number of layers

# network creation with parameters
ista_net = UnfoldedISTA(channels=Nchannels, features=Nnodes, depth=Nlayers).to(device)

print("Number of parameters :", sum(w.numel() for w in ista_net.parameters()))

Number of parameters : 10291


## Training

### Training configuration

The provided `Trainer` class allows encapsulating the network training process, and requires the following arguments:
- `model`: the network to train,
- `optimizer`: the optimization algorithm used to minimize the loss,
- `loss`: the loss function to minimize,
- `train_dataset`: the training dataset,
- `val_dataset` (optional): the validation dataset,
- `check_val_every_n_epoch` (optional): if a validation dataset is provided, this allows performing a validation step every $n$ training epochs (which helps reduce computation time).

Training also requires setting the following three parameters:
- the batch size `batch_size` (a subset of the dataset), corresponding to the number of images used before updating the model parameters,
- the number of epochs `Nepochs`, corresponding to the number of times the learning algorithm processes the entire training dataset,
- the number `NepochsToValidate` of training epochs to perform before running a validation step.

Schematically, training can be seen as a for-loop over the number of epochs, where each iteration processes the entire training dataset. Inside this loop, another loop iterates over each batch.

In [11]:
# parameters
Nepochs    = 100             # number of epochs
batch_size = 100             # size of a batch
NepochsToValidate = 10      # validation frequency

# choice of loss
loss      = torch.nn.MSELoss()
optimizer = Adam(ista_net.parameters(), lr=1e-3)

# Training initialization
ista_trainer = optim.Trainer(ista_net, optimizer, loss, batch_size,
                  train_noisybsds, val_noisybsds, check_val_every_n_epoch=NepochsToValidate)


In [12]:
ista_trainer.run(Nepochs)

Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

Validation epoch   10 | Loss: 8.99e-03 | PSNR: 2.05e+01 | Wall time: 7.64e-01
Validation epoch   20 | Loss: 5.07e-03 | PSNR: 2.30e+01 | Wall time: 7.71e-01
Validation epoch   30 | Loss: 3.42e-03 | PSNR: 2.50e+01 | Wall time: 7.53e-01
Validation epoch   40 | Loss: 3.08e-03 | PSNR: 2.55e+01 | Wall time: 7.75e-01
Validation epoch   50 | Loss: 2.88e-03 | PSNR: 2.57e+01 | Wall time: 7.74e-01
Validation epoch   60 | Loss: 2.74e-03 | PSNR: 2.60e+01 | Wall time: 7.60e-01
Validation epoch   70 | Loss: 2.60e-03 | PSNR: 2.62e+01 | Wall time: 7.57e-01
Validation epoch   80 | Loss: 2.53e-03 | PSNR: 2.64e+01 | Wall time: 7.68e-01
Validation epoch   90 | Loss: 2.46e-03 | PSNR: 2.66e+01 | Wall time: 7.55e-01
Validation epoch  100 | Loss: 2.40e-03 | PSNR: 2.66e+01 | Wall time: 7.54e-01


### Displaying training and validation performance

In [13]:
data_train = optim.getData(ista_trainer, 'Training')
data_valid = optim.getData(ista_trainer, 'Validation')        

plt_train_loss = pandas.DataFrame({'x':data_train["epoch"], 'y':data_train["Loss"], 'legend':'training loss',   'type':'loss'})
plt_valid_loss = pandas.DataFrame({'x':data_valid["epoch"], 'y':data_valid["Loss"], 'legend':'validation loss', 'type':'loss'})
plt_loss       = pandas.concat([plt_train_loss,plt_valid_loss]) 
    
figLoss = px.line(plt_loss,
            x='x', 
            y='y', 
            log_x=False, log_y=True,
            color='legend',
            labels={'x':'epochs','y':'Loss (logscale)'},
            title='Evolution of the cost function depending on epochs',
            width=800, height=450)
figLoss.show()


plt_train_psnr = pandas.DataFrame({'x':data_train["epoch"], 'y':data_train["PSNR"], 'legend':'training PSNR',   'type':'PSNR'})
plt_valid_psnr = pandas.DataFrame({'x':data_valid["epoch"], 'y':data_valid["PSNR"], 'legend':'validation PSNR', 'type':'PSNR'})
plt_psnr       = pandas.concat([plt_train_psnr,plt_valid_psnr]) 
    
figPSNR = px.line(plt_psnr,
            x='x', 
            y='y', 
            log_x=False, log_y=True,
            color='legend',
            labels={'x':'epochs','y':'PSNR (logscale)'},
            title='Evolution of PSNR depending on epochs',
            width=800, height=450)
figPSNR.show()

## Test

In [14]:
# Test image loading
z, xbar, img_id = test_noisybsds[10]

img_xbar = optim.torchImg2Numpy(xbar)
img_z    = optim.torchImg2Numpy(z)

snr_z = optim.snr(img_z, img_xbar)

# application of the network to the degarded data z
with torch.no_grad():
    xhat = ista_net(z)


img_xhat = optim.torchImg2Numpy(xhat)  
snr_xhat = optim.snr(img_xhat, img_xbar)
    
# display of the result

fig = px.imshow(np.array([img_xbar,img_z,img_xhat]), color_continuous_scale='gray',
                    title='image id : '+ str(img_id),
                    facet_col=0, facet_col_spacing=0, facet_col_wrap=3,
                    width=800, height=400,
                    )
    
fig.update_coloraxes(cmin=0, cmax=1)

legend = [  'Ground truth xbar',
            'Degraded data z (PSNR='  +"{:.2f}".format(snr_z)+'dB)',
            'Estimation xhat (PSNR='+"{:.2f}".format(snr_xhat)+'dB)'   ]
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))

fig.show()